# Improvement iteration 1: fewer input facts and grounded beam selection (dev)

**Question.** Can the M3 pipeline be improved without retraining, by acting on two Notebook 13 findings?

- **H1, distractors.** The gold-facts oracle mostly helped by removing distractor facts: 65 of its 84 gains
  were on records whose gold facts were already retrieved. Hypothesis: passing the generator fewer
  retrieved facts (top 3 or 4 instead of 5) fixes more distractor errors than it loses through lower
  recall (dev all-gold recall@3 is 80.75%, @5 86.98%).
- **H2, copy errors.** Some errors copy a number with a wrong digit. Hypothesis: among the 4 beam
  candidates, choosing the first one that executes and whose numbers all occur in the input facts fixes
  some of these.

**Decision this iteration tests.** Whether to adopt either change for the final pipeline before the
single test run, or to move on to retraining for distractor robustness.

**Protocol.** Dev only, the saved Notebook 11 checkpoint (`models/generator-t5-base-dev-v1/best`) and the
SHA-256-verified Notebook 10 dev rankings. The test split is not loaded. No annotation field is a model
input; the grounded selector reads only the input facts and the raw table. Choosing among settings on dev
makes the chosen setting's dev score optimistic.

In [1]:
from __future__ import annotations

import json
import platform
import subprocess
import time
from pathlib import Path

import pandas as pd
import torch
import transformers
from scipy.stats import binomtest
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from finevidence.data import load_finqa_split
from finevidence.data.acquisition import PINNED_SHA256, file_sha256
from finevidence.generation import (
    build_retrieved_examples,
    load_generator_config,
    load_rankings,
    score_program_prediction,
    select_grounded_candidate,
    summarize_scores,
    verify_rankings_file,
)

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
config = load_generator_config(PROJECT_ROOT / "configs" / "generator_t5_base.toml")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHECKPOINT = PROJECT_ROOT / config.model_dir / "best"
FACT_COUNTS = (5, 4, 3)
NUM_CANDIDATES = config.num_beams
torch.manual_seed(config.seed)

assert file_sha256(RAW_DIR / "dev.json") == PINNED_SHA256["dev"]
rankings_path = PROJECT_ROOT / config.rankings_path
verify_rankings_file(rankings_path, config.rankings_sha256)
dev = load_finqa_split(RAW_DIR / "dev.json")
rankings = load_rankings(rankings_path)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
print(
    f"dev records {len(dev)}; device {device}; python {platform.python_version()}; "
    f"torch {torch.__version__}; transformers {transformers.__version__}; commit {commit}"
)

/Users/divyanshi/Desktop/nlp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dev records 883; device mps; python 3.13.16; torch 2.15.0; transformers 5.22.1; commit 469e59a


## 1. Decode the top 4 beam candidates for each fact count

Beam search with 4 beams returns the 4 final candidates in score order. Candidate 0 is the ordinary
beam-search output, the same as Notebook 11 produced.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT).to(device).eval()


@torch.no_grad()
def decode_candidates(examples) -> list[list[str]]:
    candidates = []
    for start in range(0, len(examples), config.eval_batch_size):
        batch = examples[start : start + config.eval_batch_size]
        encoded = tokenizer(
            [example.source for example in batch],
            max_length=config.max_source_length,
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(device)
        output = model.generate(
            **encoded,
            max_new_tokens=config.max_target_length,
            num_beams=config.num_beams,
            num_return_sequences=NUM_CANDIDATES,
        )
        texts = tokenizer.batch_decode(
            output, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        candidates.extend(
            texts[i : i + NUM_CANDIDATES] for i in range(0, len(texts), NUM_CANDIDATES)
        )
    return candidates


examples_by_k, candidates_by_k = {}, {}
for k in FACT_COUNTS:
    started = time.time()
    examples_by_k[k] = build_retrieved_examples(dev, rankings, facts_per_input=k)
    candidates_by_k[k] = decode_candidates(examples_by_k[k])
    print(
        f"top {k} facts: decoded {len(candidates_by_k[k])} records in {time.time() - started:.0f}s"
    )

W1101 22:16:26.471000 53580 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 257/257 [00:00<00:00, 18624.28it/s]

top 5 facts: decoded 883 records in 22158s


top 4 facts: decoded 883 records in 18691s


top 3 facts: decoded 883 records in 264s


In [3]:
saved = {
    json.loads(line)["record_id"]: json.loads(line)["prediction"]
    for line in (PROCESSED_DIR / f"{config.experiment_name}_dev_retrieved_predictions.jsonl")
    .read_text()
    .splitlines()
}
same = sum(
    candidates[0] == saved[example.record_id]
    for example, candidates in zip(examples_by_k[5], candidates_by_k[5], strict=True)
)
print(f"top-5, candidate 0 identical to the saved Notebook 11 prediction: {same}/{len(dev)}")

top-5, candidate 0 identical to the saved Notebook 11 prediction: 883/883


## 2. Score top beam and grounded selection

Both selections are scored with the same official execution-accuracy code as Notebook 11.

In [4]:
results, scores_by_setting = {}, {}
for k in FACT_COUNTS:
    for selection in ("top_beam", "grounded"):
        scores = []
        for record, example, candidates in zip(
            dev, examples_by_k[k], candidates_by_k[k], strict=True
        ):
            index = (
                0
                if selection == "top_beam"
                else select_grounded_candidate(candidates, example.fact_texts, record["table"])
            )
            scores.append(score_program_prediction(record, candidates[index], example.fact_texts))
        setting = f"top {k} facts, {selection}"
        scores_by_setting[setting] = scores
        results[setting] = summarize_scores(scores)
summary = pd.DataFrame(results).T
summary

,records,execution_correct,execution_accuracy,program_accuracy,invalid_rate,operand_grounding_rate
"top 5 facts, top_beam",883.0,523.0,0.592299,0.556059,0.007928,0.970436
"top 5 facts, grounded",883.0,547.0,0.619479,0.585504,0.001133,0.992256
"top 4 facts, top_beam",883.0,516.0,0.584371,0.550396,0.007928,0.969933
"top 4 facts, grounded",883.0,542.0,0.613817,0.582106,0.002265,0.992232
"top 3 facts, top_beam",883.0,509.0,0.576444,0.540204,0.009060,0.969776
"top 3 facts, grounded",883.0,537.0,0.608154,0.574179,0.004530,0.994816


## 3. Paired comparison with the M3 baseline (top 5 facts, top beam)

Discordant records: fixed means the baseline was wrong and the setting right; broken is the reverse.
The p-value is an exact two-sided McNemar test (a binomial test on the discordant records).

In [5]:
baseline = scores_by_setting["top 5 facts, top_beam"]
rows = {}
for setting, scores in scores_by_setting.items():
    fixed = sum(
        not b.execution_correct and s.execution_correct
        for b, s in zip(baseline, scores, strict=True)
    )
    broken = sum(
        b.execution_correct and not s.execution_correct
        for b, s in zip(baseline, scores, strict=True)
    )
    p_value = binomtest(fixed, fixed + broken).pvalue if fixed + broken else 1.0
    rows[setting] = {
        "fixed": fixed,
        "broken": broken,
        "net": fixed - broken,
        "p_value": round(p_value, 4),
    }
pd.DataFrame(rows).T

,fixed,broken,net,p_value
"top 5 facts, top_beam",0.0,0.0,0.0,1.0000
"top 5 facts, grounded",24.0,0.0,24.0,0.0000
"top 4 facts, top_beam",8.0,15.0,-7.0,0.2100
"top 4 facts, grounded",29.0,10.0,19.0,0.0034
"top 3 facts, top_beam",13.0,27.0,-14.0,0.0385
"top 3 facts, grounded",35.0,21.0,14.0,0.0814


In [6]:
grounded_changes = sum(
    select_grounded_candidate(c, e.fact_texts, r["table"]) != 0
    for r, e, c in zip(dev, examples_by_k[5], candidates_by_k[5], strict=True)
)
print(
    f"top 5 facts: grounded selection changed the chosen candidate for {grounded_changes} records"
)

out = PROCESSED_DIR / f"{config.experiment_name}_dev_iteration1_candidates.jsonl"
with out.open("w", encoding="utf-8") as handle:
    for k in FACT_COUNTS:
        for example, candidates in zip(examples_by_k[k], candidates_by_k[k], strict=True):
            row = {
                "record_id": example.record_id,
                "facts": k,
                "fact_keys": example.fact_keys,
                "candidates": candidates,
                "commit": commit,
            }
            handle.write(json.dumps(row) + "\n")
print(out.relative_to(PROJECT_ROOT))

top 5 facts: grounded selection changed the chosen candidate for 47 records
data/processed/generator-t5-base-dev-v1_dev_iteration1_candidates.jsonl


## 4. Interpretation (from the outputs above)

**Reproduction.** With top 5 facts, candidate 0 is identical to the saved Notebook 11 prediction for all
883 records, so local decoding (MPS, torch 2.15, transformers 5.22) reproduces the Kaggle run, and the
baseline row (523 correct, 59.23%) matches Notebook 11.

**H1 is rejected: fewer facts hurt.** With top-beam selection, top 4 facts gives 516 correct (8 fixed, 15
broken) and top 3 gives 509 (13 fixed, 27 broken, p = 0.039). Losing gold facts costs more than removing
distractors saves. Distractor robustness will need a training-side change, not a smaller input.

**H2 is supported: grounded selection helps.** With top 5 facts, choosing the first beam candidate that
executes and uses only numbers present in the facts gives **547 correct (61.95%)** against 523 (59.23%):
24 records fixed and **0 broken** (exact McNemar p ≈ 1.2e-7). It changed the chosen candidate for 47
records. Program accuracy rises from 55.61% to 58.55%, the invalid rate falls from 0.79% to 0.11%, and
operand grounding from 97.0% to 99.2%. Grounded selection also helps with 4 and 3 facts, but less than
with 5.

**Decision.** Adopt **top 5 facts + grounded beam selection** as the current pipeline. It has no tuned
parameter beyond choosing among these 6 settings, and it broke no dev record, so its gain is unlikely to be
dev-selection noise. It still has to be confirmed in the single test run.

**Limitations.** Dev only; one checkpoint and seed. The wall-clock decode times above (about 6 h and 5 h
for the first two settings, 4 min for the third) show the machine was suspended during the run, so they
are not latency measurements; per-question latency stays an M4 inference-record item.

**Next iteration.** Distractor robustness by retraining: train the generator with retrieved-style hard
distractors (top-ranked non-gold facts) instead of random ones. That needs the Kaggle GPU (about 2 h).